# Chapter 11: check the KPI dictionary file

Section 11.6 reads a dictionary CSV and fails any entry with a missing field, a formula that names a column the
model lacks or an owner left blank. Names written in backticks in the formula or the source are checked against
the tables and columns of Appendix C.

In [ ]:
import sys
sys.path.insert(0, "..")
import re
import pandas as pd
import kestrel as k

FIELDS = ["name", "plain_definition", "formula", "grain", "source", "owner", "version", "check_value"]
model_names = set()
for t in k.CORE + k.REFERENCE:
    model_names.add(t)
    model_names.update(k.load(t).columns)
model_names.update(["node", "edge", "lineage_path", "source_comparison", "quarantine"])

dictionary = pd.read_csv(k.DATA / "kpi_dictionary.csv")
print(dictionary[["name", "grain", "owner", "version"]].to_string(index=False))

In [ ]:
def check(dictionary):
    problems = []
    for _, e in dictionary.iterrows():
        for f in FIELDS:
            if pd.isna(e.get(f)) or not str(e.get(f)).strip():
                problems.append((e["name"], f"missing {f}"))
        for f in ("formula", "source"):
            for ref in re.findall(r"`([^`]+)`", str(e.get(f, ""))):
                if ref not in model_names:
                    problems.append((e["name"], f"{f} names `{ref}`, which the model lacks"))
    names = dictionary.name.value_counts()
    problems += [(n, "name used twice") for n in names[names > 1].index]
    return pd.DataFrame(problems, columns=["entry", "problem"])

result = check(dictionary)
print("dictionary passes" if result.empty else result.to_string(index=False))

Now break a copy on purpose: an owner left blank, a formula naming a column nobody built and a missing grain.

In [ ]:
broken = dictionary.copy()
broken.loc[5, "owner"] = " "
broken.loc[1, "formula"] = "`cast_volume_l` / `grist_kg` x 100"
broken.loc[12, "grain"] = None
print(check(broken).to_string(index=False))